In [1]:
import numpy as np
import pandas as pd

np.random.seed(42)

In [5]:
users = pd.read_csv("users.csv")

In [2]:
n_transactions = 800000

In [3]:
transaction_ids = np.arange(1, n_transactions + 1)

In [6]:
user_ids_txn = np.random.choice(
    users["user_id"],
    size=n_transactions)

In [7]:
transaction_dates = np.random.choice(
    pd.date_range("2024-01-01", "2025-01-01"),
    size=n_transactions
)

In [8]:
amounts = np.random.lognormal(
    mean=4.0,
    sigma=1.0,
    size=n_transactions
)

In [9]:
merchant_categories = np.random.choice(
    ["retail", "travel", "food", "electronics", "subscription"],
    size=n_transactions,
    p=[0.35, 0.15, 0.25, 0.15, 0.10]
)

In [10]:
device_risk_score = np.random.beta(2, 8, size=n_transactions)

In [11]:
geo_risk_score = np.random.beta(2, 6, size=n_transactions)

In [12]:
transactions = pd.DataFrame({
    "transaction_id": transaction_ids,
    "user_id": user_ids_txn,
    "transaction_date": transaction_dates,
    "amount": amounts,
    "merchant_category": merchant_categories,
    "device_risk_score": device_risk_score,
    "geo_risk_score": geo_risk_score
})

In [13]:
transactions = transactions.merge(
    users[["user_id", "tenure_days"]],
    on="user_id",
    how="left"
)

In [14]:
fraud_probability = (
    0.002 +
    0.02 * transactions["device_risk_score"] +
    0.015 * transactions["geo_risk_score"] +
    0.00001 * transactions["amount"] +
    0.005 * (transactions["tenure_days"] < 30)
)

In [15]:
fraud_probability = np.clip(fraud_probability, 0, 1)

In [16]:
transactions["is_fraud"] = np.random.binomial(
    1,
    fraud_probability
)

In [17]:
transactions = transactions.drop(columns=["tenure_days"])

In [18]:
transactions.head()

,transaction_id,user_id,transaction_date,amount,merchant_category,device_risk_score,geo_risk_score,is_fraud
0,1,15796,2024-04-22,31.325697,electronics,0.450375,0.452995,0
1,2,861,2024-08-26,7.937016,food,0.295916,0.178344,0
2,3,38159,2024-10-10,64.424860,retail,0.215217,0.180407,0
3,4,44733,2024-09-14,36.397554,retail,0.242987,0.150698,0
4,5,11285,2024-04-17,13.315346,retail,0.160120,0.225078,0


In [19]:
transactions["is_fraud"].mean()

0.01095875

In [20]:
transactions.to_csv("transactions.csv", index=False)